# ___Updating the photosynthetic pathways___
--------------------

In [1]:
!python --version

Python 3.14.2


The system cannot find the path specified.


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# https://www.tern.org.au/news/news-photosynthetic-pathways/
# https://portal.tern.org.au/metadata/TERN/1e16257d-57ae-48dd-bbad-5701e72a9f6d
# TERN is an Australia specific dataset
tern = pd.read_csv(r"../../data/chapter2/TERN/Photosynthetic_Pathways_of_Plants_TERN_v2_19092024.csv", encoding="latin1", 
        usecols=["genus", "speciesEpithet", "family", "photosyntheticPathway_confirmed", "photosyntheticPathway_inferred", "photosyntheticPathway_combined"], na_values='U')#, index_col=["genus", "speciesEpithet"])
tern_meta = pd.read_excel(r"../../data/chapter2/TERN/metadata_Photosynthetic_Pathways_of_Plants_TERN_v2_2024.xlsx", sheet_name="Data_Descriptor")
# extra spaces
tern.loc[:, "genus"] = tern.genus.str.strip()
tern.loc[:, "speciesEpithet"] = tern.speciesEpithet.str.strip()
tern.insert(loc=0, column="binominal", value=tern.genus.str.strip() + ' ' + tern.speciesEpithet.str.strip())

# in TRY, photosynthesis pathway is trait id 22
try_photo = pd.read_csv(r"../../data/chapter2/TRY/photosynthetic_pathways.txt", delimiter='\t', encoding="latin1", low_memory=False, decimal='.', usecols=["Dataset", "SpeciesName",
                        "AccSpeciesName", "OrigValueStr", "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"]) # records where TraitId is NaN are mostly messy metadata rows
# extra spaces 
try_photo.loc[:, "SpeciesName"] = try_photo.SpeciesName.str.strip()
try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.str.upper().str.replace('.', '')
# messy photosynthetic pathway information
PHOTOSYNTHETIC_PATHWAY_TYPES = { # try to make this as clean as possible!!
    "C3": "C3",
    "C3?": "C3?",
    "C4": "C4",
    "C4?": "C4?",
    "CAM": "CAM",
    "CAM?": "CAM?",
    "C3/C4": "C3/C4",
    "C3C4": "C3/C4",
    "C3/CAM": "C3/CAM",
    "C3-CAM": "C3/CAM",
    "C4/CAM": "C4/CAM",
    "C4-CAM": "C4/CAM",
    "C3/C4/CAM": "C3/C4/CAM",
    "3": "C3"
}
try_photo.loc[:, "OrigValueStr"] = try_photo.OrigValueStr.replace(PHOTOSYNTHETIC_PATHWAY_TYPES) # clean up the irregularities in the column

collab_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_collab_categorical.csv")

# even if missing in our subset, FRED may still contain photosynthetic pathway information of the desired species
fred_photo = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1",
                         usecols=("F01286", "F01287", "F00043", "F00004")).dropna(subset=("F01286", "F01287", "F00043")).drop_duplicates()
fred_photo.insert(loc=0, column="binominal", value=fred_photo.F01286.str.strip().str.capitalize() + ' ' + fred_photo.F01287.str.strip().str.lower())

# GRoot was not considered for mycorrhizal states as most of its mycorrhizal state info came from FungalRoot which was obe of the dataset that was used in populating the missing fields
groot = pd.read_csv(r"../../data/chapter2/GRooTFullVersion.csv", encoding="latin1", low_memory=False)
groot.insert(loc=0, column="binominal", value=groot.genus.str.capitalize().str.strip() + ' ' + groot.species.str.lower().str.strip())

In [20]:
# this is the list of species we're interested in finding the photosynthetic pathways for
species_of_interest = collab_categorical.loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True) 
species_of_interest

0              Acer saccharum
1          Fraxinus americana
2             Viola pubescens
3      Hydrophyllum canadense
4              Larix gmelinii
                ...          
390            Acer triflorum
391          Pinus massoniana
392           Malus domestica
393            Prunus persica
394            Vitis vinifera
Length: 395, dtype: object

### ___TERN___
-------------------

In [37]:
tern_ = tern.query("binominal.isin(@species_of_interest)") # that's disappointing
tern_

,binominal,genus,speciesEpithet,family,photosyntheticPathway_confirmed,photosyntheticPathway_inferred,photosyntheticPathway_combined
250,Acacia auriculiformis,Acacia,auriculiformis,Fabaceae,C3,NaN,C3
968,Galium aparine,Galium,aparine,Rubiaceae,C3,NaN,C3
2179,Acacia crassicarpa,Acacia,crassicarpa,Fabaceae,NaN,C3,C3


### ___FRED___
------------------

In [33]:
subset_missing_pathway_species = collab_categorical.query("F00043.isna()").loc[:, ["F01286", "F01287"]].drop_duplicates().agg(' '.join, axis=1).str.strip().reset_index(drop=True)
subset_missing_pathway_species

0         Euonymus verrucosus
1               Alnus hirsuta
2             Alnus formosana
3       Cinnamomum micranthum
4         Altingia gracilipes
               ...           
94           Osmunda japonica
95        Dioscorea nipponica
96    Cotoneaster multiflorus
97           Adina pilulifera
98         Castanopsis faberi
Length: 99, dtype: object

In [38]:
# since this is FRED, look for species that we do not have info (nans) for in the subset
fred_photo_ = fred_photo.query("binominal.isin(@subset_missing_pathway_species)").drop_duplicates() # that's nice
fred_photo_

,binominal,F00004,F01286,F01287,F00043
15,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",Gordonia,axillaris,C3
842,Castanopsis faberi,"Xiong D, Huang J, Yang Z, Lu Z, Chen G, Yang Y...",Castanopsis,faberi,C3
8218,Aesculus hippocastanum,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",Aesculus,hippocastanum,C3
9730,Gordonia axillaris,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",Gordonia,axillaris,C3
9736,Psychotria asiatica,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",Psychotria,asiatica,C3
24726,Sanguisorba officinalis,"Cheng J, Chu P, Chen D, Bai Y. 2016. Functiona...",Sanguisorba,officinalis,C3
31436,Castanopsis faberi,"Lin C, Yang Y, Guo J, Chen G, Xie J. 2011. Fin...",Castanopsis,faberi,C3
48154,Aesculus hippocastanum,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Aesculus,hippocastanum,C3
52287,Rosa acicularis,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Rosa,acicularis,C3
52440,Sanguisorba officinalis,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",Sanguisorba,officinalis,C3


In [35]:
# all of them are C3 but anyways

### ___TRY___
-----------------------

In [41]:
try_photo_ = try_photo.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates(subset=["AccSpeciesName", "OrigValueStr"])
try_photo_

,Dataset,SpeciesName,AccSpeciesName,TraitID,OrigValueStr
645,Sheffield-Iran-Spain Database,Acer pseudoplatanus,Acer pseudoplatanus,22.0,C3
1529,Sheffield-Iran-Spain Database,Fraxinus excelsior,Fraxinus excelsior,22.0,C3
1546,Sheffield-Iran-Spain Database,Galium aparine,Galium aparine,22.0,C3
2107,Sheffield-Iran-Spain Database,Pinus sylvestris,Pinus sylvestris,22.0,C3
2277,Sheffield-Iran-Spain Database,Pteridium aquilinum,Pteridium aquilinum,22.0,C3
...,...,...,...,...,...
2123927,TRY Categorical Traits Dataset (update 2018),Cyclobalanopsis oxyodon,Quercus oxyodon,22.0,C3
2126294,TRY Categorical Traits Dataset (update 2018),Lindera obtusiloba,Lindera obtusiloba,22.0,C3
2130901,TRY Categorical Traits Dataset (update 2018),Tilia oliveri,Tilia oliveri,22.0,C3
2151157,TRY Categorical Traits Dataset (update 2018),Allophylus occidentalis,Allophylus cobbe,22.0,C3


In [42]:
try_photo_.query("AccSpeciesName.isin(@species_of_interest)").drop_duplicates().OrigValueStr.unique()

array(['C3', 'C4', 'C3/CAM', 'C3?', 'UNKNOWN'], dtype=object)

### ___GRoot___
--------------------------

In [45]:
groot_ = groot.query("not photosyntheticPathway.isna() and binominal.isin(@species_of_interest)").loc[:, ["binominal", "photosyntheticPathway", "references", "referencesDataset"]].\
    drop_duplicates(subset=["binominal", "photosyntheticPathway"])
groot_

,binominal,photosyntheticPathway,references,referencesDataset
11,Acer saccharum,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
18,Pinus strobus,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
21,Quercus alba,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
24,Quercus rubra,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
28,Pinus resinosa,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Gill, R., and R. B. Jackson. 2003. Global Dist..."
...,...,...,...,...
58648,Saussurea mongolica,C3,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN
58684,Sorbus discolor,C3,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN
59659,Altingia obovata,C3,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",NaN
81541,Archidendron clypearia,C3,"Valverde-Barrantes O J, Horning A L, Smemo K A...",NaN


In [46]:
groot_.query("not photosyntheticPathway.isna() and binominal.isin(@species_of_interest)").loc[:, ["binominal", "photosyntheticPathway", "references", "referencesDataset"]].drop_duplicates().photosyntheticPathway.unique()

array(['C3', 'C3/C4'], dtype=object)

In [57]:
#-----------------------------------------------------------------------------------------------------
# COMBINE ALL OF THESE AND CREATE A SINGLE DATASET
#-----------------------------------------------------------------------------------------------------

data = pd.merge(left=tern_, left_on="binominal", right=fred_photo_, right_on="binominal", how="outer").loc[:, ["binominal", "F00004", "F00043", "photosyntheticPathway_combined"]]
data

,binominal,F00004,F00043,photosyntheticPathway_combined
0,Acacia auriculiformis,NaN,NaN,C3
1,Acacia crassicarpa,NaN,NaN,C3
2,Aesculus hippocastanum,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",C3,NaN
3,Aesculus hippocastanum,"Akhmetzhanova AA, Soudzilovskaiana NA, Onipche...",C3,NaN
4,Castanopsis faberi,"Xiong D, Huang J, Yang Z, Lu Z, Chen G, Yang Y...",C3,NaN
5,Castanopsis faberi,"Lin C, Yang Y, Guo J, Chen G, Xie J. 2011. Fin...",C3,NaN
6,Castanopsis faberi,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",C3,NaN
7,Galium aparine,NaN,NaN,C3
8,Gordonia axillaris,"Long Y, Kong D, Chen Z, Zeng H. 2013. Variatio...",C3,NaN
9,Gordonia axillaris,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",C3,NaN


In [65]:
data = pd.merge(left=data, left_on="binominal", right=try_photo_.rename({"AccSpeciesName": "binominal"}, axis=1), right_on="binominal", how="outer").drop(["SpeciesName", "TraitID"], axis=1)
data

,binominal,F00004,F00043,photosyntheticPathway_combined,Dataset,OrigValueStr
0,Abies fargesii,NaN,NaN,NaN,Categorical Plant Traits Database,C3
1,Abies nephrolepis,NaN,NaN,NaN,Categorical Plant Traits Database,C3
2,Acacia auriculiformis,NaN,NaN,C3,Categorical Plant Traits Database,C3
3,Acacia crassicarpa,NaN,NaN,C3,FRED - Fine Root Ecology Database,C3
4,Acacia mangium,NaN,NaN,NaN,Categorical Plant Traits Database,C3
...,...,...,...,...,...,...
312,Veratrum nigrum,NaN,NaN,NaN,Categorical Plant Traits Database,C3
313,Veronica spuria,NaN,NaN,NaN,BIOPOP: Functional Traits for Nature Conservation,C3
314,Viola pubescens,NaN,NaN,NaN,Categorical Plant Traits Database,C3
315,Vitis amurensis,NaN,NaN,NaN,The China Plant Trait Database,C3


In [67]:
data = pd.merge(left=data, left_on="binominal", right=groot_, right_on="binominal", how="outer")
data

,binominal,F00004,F00043,photosyntheticPathway_combined,Dataset,OrigValueStr,photosyntheticPathway,references,referencesDataset
0,Abelia biflora,NaN,NaN,NaN,NaN,NaN,C3,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN
1,Abies fargesii,NaN,NaN,NaN,Categorical Plant Traits Database,C3,NaN,NaN,NaN
2,Abies nephrolepis,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Dong L, Mao Z, Sun T. 2016. Condensed tannin e...",NaN
3,Acacia auriculiformis,NaN,NaN,C3,Categorical Plant Traits Database,C3,C3,"Das DK, Chaturvedi OP. 2008. Root phytomass re...","Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,..."
4,Acacia crassicarpa,NaN,NaN,C3,FRED - Fine Root Ecology Database,C3,C3,"Kong D, Ma C, Zhang Q, Li L, Chen X, Zeng H, G...",NaN
...,...,...,...,...,...,...,...,...,...
317,Veratrum nigrum,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",NaN
318,Veronica spuria,NaN,NaN,NaN,BIOPOP: Functional Traits for Nature Conservation,C3,NaN,NaN,NaN
319,Viola pubescens,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",NaN
320,Vitis amurensis,NaN,NaN,NaN,The China Plant Trait Database,C3,NaN,NaN,NaN


In [ ]:
# here
# F00004, F00043 ARE FROM FRED V3
# photosyntheticPathway_combined IS FROM TERN
# Dataset, OrigValueStr ARE FROM TRY
# photosyntheticPathway, references & referencesDataset ARE FROM GRoot

In [68]:
# serialize it :)
data.to_csv(r"../../data/chapter2/FREDv3subset/pathways_to_fill_FRED_TRY_TERN_GRoot.csv", index=False)

### ___Conflict resolution & population___
-------------------------

In [14]:
lookup = pd.read_csv(r"../../data/chapter2/FREDv3subset/pathways_to_fill_FRED_TRY_TERN_GRoot.csv")

In [8]:
collab_categorical

,F01286,F01287,F01289,F01290,F00004,F00043,F00645
0,Acer,saccharum,Sapindaceae,Sapindales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
1,Fraxinus,americana,Oleaceae,Lamiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
2,Viola,pubescens,Violaceae,Malpighiales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
3,Hydrophyllum,canadense,Boraginaceae,Boraginales,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",C3,NaN
4,Larix,gmelinii,Pinaceae,Pinales,"Wang Z, Guo D, Wang X, Gu J, Mei L. 2006. Fine...",C3,NaN
...,...,...,...,...,...,...,...
542,Pinus,koraiensis,Pinaceae,Pinales,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",C3,NaN
543,Pinus,sylvestris,Pinaceae,Pinales,"Wang Y, Gao G, Wang N, Wang Z, Gu J. 2019. Eff...",C3,NaN
544,Malus,domestica,Rosaceae,Rosales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN
545,Prunus,persica,Rosaceae,Rosales,"Lavely EL, Chen W, Peterson KA, Klodd AE, Vold...",C3,NaN


In [16]:
collab_pathways = pd.Series(index=collab_categorical.F01286 + ' ' + collab_categorical.F01287, data=collab_categorical.F00043.values, name="F00043")
collab_pathways

Acer saccharum            C3
Fraxinus americana        C3
Viola pubescens           C3
Hydrophyllum canadense    C3
Larix gmelinii            C3
                          ..
Pinus koraiensis          C3
Pinus sylvestris          C3
Malus domestica           C3
Prunus persica            C3
Vitis vinifera            C3
Name: F00043, Length: 547, dtype: object

In [22]:
merged = pd.merge(left=collab_pathways, left_index=True, right=lookup, right_on="binominal", suffixes=("_", None), how="left")
merged

,F00043_,binominal,F00004,F00043,photosyntheticPathway_combined,Dataset,OrigValueStr,photosyntheticPathway,references,referencesDataset
17.0,C3,Acer saccharum,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
138.0,C3,Fraxinus americana,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Aulen M, Shipley B, Bradley R. 2012. Predictio...",NaN
319.0,C3,Viola pubescens,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",NaN
154.0,C3,Hydrophyllum canadense,NaN,NaN,NaN,FRED - Fine Root Ecology Database,C3,C3,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",NaN
163.0,C3,Larix gmelinii,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Bai, H.M., Li, G.T., Yue, Y.J., Li, Y.J. & Ma,...","Wang C, Chen Z, Brunner I, Zhang Z, Zhu X, Li ..."
...,...,...,...,...,...,...,...,...,...,...
235.0,C3,Pinus sylvestris,NaN,NaN,NaN,Sheffield-Iran-Spain Database,C3,C3,"Ahlstrom K, Persson H, Borjesson I. 1988. Fer...","Gordon, W.S., and R.B. Jackson. 2003. Global d..."
236.0,C3,Pinus sylvestris,NaN,NaN,NaN,Categorical Plant Traits Database,C3/CAM,C3,"Ahlstrom K, Persson H, Borjesson I. 1988. Fer...","Gordon, W.S., and R.B. Jackson. 2003. Global d..."
205.0,C3,Malus domestica,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Bouma TJ, Yanai RD, Elkin AD, Hartmond U, Flor...",NaN
254.0,C3,Prunus persica,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,Howard A. 1925. The effects of grass on trees....,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,..."


In [27]:
# first step - look for conflicts between the records our subset has pathway info for and the lookup dataset
merged.dropna(subset=["F00043_"], axis=0) # F00043_ is the column from the orginal FRED subset
# we have four columns to check conflicts => F00004, photosyntheticPathway_combined (TERN), OrigValueStr (TRY) & photosyntheticPathway (GRoot)

,F00043_,binominal,F00004,F00043,photosyntheticPathway_combined,Dataset,OrigValueStr,photosyntheticPathway,references,referencesDataset
17.0,C3,Acer saccharum,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Aber JD, Melillo JM, Nadelhoffer KJ, McClaughe...","Nadelhoffer KJ, Raich JW. 1992. Fine root prod..."
138.0,C3,Fraxinus americana,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Aulen M, Shipley B, Bradley R. 2012. Predictio...",NaN
319.0,C3,Viola pubescens,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",NaN
154.0,C3,Hydrophyllum canadense,NaN,NaN,NaN,FRED - Fine Root Ecology Database,C3,C3,"Pregitzer KS, Kubiske ME, Yu CK, Hendrick RL. ...",NaN
163.0,C3,Larix gmelinii,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Bai, H.M., Li, G.T., Yue, Y.J., Li, Y.J. & Ma,...","Wang C, Chen Z, Brunner I, Zhang Z, Zhu X, Li ..."
...,...,...,...,...,...,...,...,...,...,...
235.0,C3,Pinus sylvestris,NaN,NaN,NaN,Sheffield-Iran-Spain Database,C3,C3,"Ahlstrom K, Persson H, Borjesson I. 1988. Fer...","Gordon, W.S., and R.B. Jackson. 2003. Global d..."
236.0,C3,Pinus sylvestris,NaN,NaN,NaN,Categorical Plant Traits Database,C3/CAM,C3,"Ahlstrom K, Persson H, Borjesson I. 1988. Fer...","Gordon, W.S., and R.B. Jackson. 2003. Global d..."
205.0,C3,Malus domestica,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,"Bouma TJ, Yanai RD, Elkin AD, Hartmond U, Flor...",NaN
254.0,C3,Prunus persica,NaN,NaN,NaN,Categorical Plant Traits Database,C3,C3,Howard A. 1925. The effects of grass on trees....,"Fan Y, Miguez-Macho G, Jobbagy EG, Jackson RB,..."


In [34]:
for (_, row) in merged.dropna(subset=["F00043_"], axis=0).iterrows():
    if row["F00004"] != row["photosyntheticPathway_combined"] != row["OrigValueStr"] != row["photosyntheticPathway"]:
        # print(row["F00004_", "F00004", "photosyntheticPathway_combined", "OrigValueStr", "photosyntheticPathway"])

KeyError: 'key of type tuple not found and not a MultiIndex'